# 📘 CIFAR-10 Image Classification: ANN vs CNN
## Complete, Improved & Fixed Version

This notebook covers the **complete deep learning pipeline**:
- Data loading, visualization & preprocessing
- ANN model (baseline)
- CNN model (improved)
- CNN + Data Augmentation (best)
- Full comparison with plots

🎯 **Goal:** Understand why CNN outperforms ANN for image classification.

## 📦 Step 1: Install & Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Fix random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

## 📥 Step 2: Load CIFAR-10 Dataset

CIFAR-10 has **60,000 color images** of size **32×32×3** across **10 classes**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Train images shape:', x_train.shape)   # (50000, 32, 32, 3)
print('Train labels shape:', y_train.shape)   # (50000, 1)
print('Test images shape: ', x_test.shape)    # (10000, 32, 32, 3)
print('Test labels shape: ', y_test.shape)    # (10000, 1)
print('Pixel value range: ', x_train.min(), '-', x_train.max())

## 🖼️ Step 3: Visualize Sample Images

In [ ]:
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]], fontsize=10)
    plt.axis('off')
plt.suptitle('Sample CIFAR-10 Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Show class distribution
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 3))
plt.bar(class_names, counts, color='steelblue', edgecolor='black')
plt.title('Class Distribution in Training Set')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print('Classes are balanced:', np.all(counts == counts[0]))

## 🧹 Step 4: Preprocessing

**Why normalize?** Raw pixel values are 0–255. Normalizing to 0–1:
- Speeds up training (gradients are more stable)
- Prevents one feature from dominating

**ANN** needs images **flattened** (32×32×3 = 3072 values per image).  
**CNN** keeps the **2D spatial structure** (32×32×3).

In [ ]:
# Normalize pixel values: 0-255 → 0.0-1.0
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm  = x_test.astype('float32')  / 255.0

# Flatten for ANN: (50000, 32, 32, 3) → (50000, 3072)
x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat  = x_test_norm.reshape(len(x_test_norm), -1)

print('Normalized train (CNN input):', x_train_norm.shape)
print('Flattened train  (ANN input):', x_train_flat.shape)
print('Pixel range after norm:', x_train_norm.min(), '-', x_train_norm.max())

## 🔹 Part 1: ANN (Artificial Neural Network)

### How ANN handles images:
- Flattens the 32×32×3 image into **3072 numbers** — all spatial info is lost
- Passes through Dense layers to learn patterns
- **Limitation:** Cannot detect edges, textures, or shapes since it ignores pixel positions

### Architecture:
```
Input (3072) → Dense(512, ReLU) → Dropout(0.3) → Dense(256, ReLU) → Dense(10, Softmax)
```

In [ ]:
def build_ann():
    model = models.Sequential([
        layers.Input(shape=(3072,)),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ], name='ANN')
    return model

ann_model = build_ann()

ann_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()

In [ ]:
# Early stopping to prevent overfitting
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# Reduce LR when plateau
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    verbose=1
)

print('Training ANN...')
ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test, verbose=0)
print(f'ANN Test Accuracy : {ann_test_acc:.4f} ({ann_test_acc*100:.2f}%)')
print(f'ANN Test Loss     : {ann_test_loss:.4f}')

## 🔹 Part 2: CNN (Convolutional Neural Network)

### How CNN handles images:
- **Conv2D** layers slide filters over the image to detect local patterns (edges, textures, shapes)
- **BatchNormalization** stabilizes training after each conv layer
- **MaxPooling2D** reduces spatial dimensions while keeping important features
- Hierarchical feature extraction: low layers → edges, high layers → objects

### Architecture:
```
Input (32,32,3)
  → Conv2D(32) + BN + MaxPool
  → Conv2D(64) + BN + MaxPool
  → Conv2D(128)
  → Flatten → Dense(128) → Dropout(0.4) → Dense(10, Softmax)
```

In [ ]:
def build_cnn():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(10, activation='softmax')
    ], name='CNN')
    return model

cnn_model = build_cnn()

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
early_stop_cnn = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_cnn = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print('Training CNN...')
cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=30,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_cnn, reduce_lr_cnn],
    verbose=1
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f'CNN Test Accuracy : {cnn_test_acc:.4f} ({cnn_test_acc*100:.2f}%)')
print(f'CNN Test Loss     : {cnn_test_loss:.4f}')

## 🚀 Part 3: CNN + Data Augmentation (Best Model)

**Data Augmentation** artificially increases training data diversity by randomly:
- Flipping images horizontally
- Rotating slightly
- Zooming in/out

This forces the model to generalize better and reduces overfitting.

In [ ]:
# Visualize augmentation effects
sample_img = x_train_norm[:1]  # single image batch

augment_layer = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

plt.figure(figsize=(12, 3))
plt.subplot(1, 6, 1)
plt.imshow(sample_img[0])
plt.title('Original')
plt.axis('off')
for i in range(5):
    aug = augment_layer(sample_img, training=True)
    plt.subplot(1, 6, i + 2)
    plt.imshow(aug[0])
    plt.title(f'Aug {i+1}')
    plt.axis('off')
plt.suptitle('Data Augmentation Examples', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
def build_aug_cnn():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),

        # Augmentation (only active during training)
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),

        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ], name='CNN_Augmented')
    return model

aug_cnn_model = build_aug_cnn()

aug_cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

aug_cnn_model.summary()

In [ ]:
early_stop_aug = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

reduce_lr_aug = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

print('Training CNN with Data Augmentation...')
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=50,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stop_aug, reduce_lr_aug],
    verbose=1
)

In [ ]:
aug_test_loss, aug_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test, verbose=0)
print(f'Aug CNN Test Accuracy : {aug_test_acc:.4f} ({aug_test_acc*100:.2f}%)')
print(f'Aug CNN Test Loss     : {aug_test_loss:.4f}')

## 📈 Step 5: Compare Learning Curves

In [ ]:
def plot_history(histories, labels, metric='accuracy'):
    """Plot training and validation curves for multiple models."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for i, (hist, label) in enumerate(zip(histories, labels)):
        c = colors[i]
        val_key = f'val_{metric}'
        if val_key in hist.history:
            axes[0].plot(hist.history[metric],       color=c, linestyle='--', alpha=0.5, label=f'{label} Train')
            axes[0].plot(hist.history[val_key],      color=c, label=f'{label} Val')
            axes[1].plot(hist.history['val_loss'],   color=c, label=f'{label} Val Loss')

    axes[0].set_title('Validation Accuracy Comparison', fontsize=13)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title('Validation Loss Comparison', fontsize=13)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(
    [ann_history, cnn_history, aug_history],
    ['ANN', 'CNN', 'CNN + Augmentation']
)

## 🔍 Step 6: Predictions & Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Use best model (aug_cnn) for predictions
y_pred_probs = aug_cnn_model.predict(x_test_norm, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = y_test.flatten()

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Confusion Matrix — CNN + Augmentation', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Visualize some correct and incorrect predictions
correct_idx   = np.where(y_pred == y_true)[0][:5]
incorrect_idx = np.where(y_pred != y_true)[0][:5]

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i, idx in enumerate(correct_idx):
    axes[0, i].imshow(x_test_norm[idx])
    axes[0, i].set_title(
        f'True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}',
        color='green', fontsize=9
    )
    axes[0, i].axis('off')

for i, idx in enumerate(incorrect_idx):
    axes[1, i].imshow(x_test_norm[idx])
    axes[1, i].set_title(
        f'True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}',
        color='red', fontsize=9
    )
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Correct ✅', fontsize=11)
axes[1, 0].set_ylabel('Incorrect ❌', fontsize=11)

plt.suptitle('Sample Predictions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 Step 7: Final Comparison Table

In [ ]:
# Count parameters in each model
ann_params = ann_model.count_params()
cnn_params = cnn_model.count_params()
aug_params = aug_cnn_model.count_params()

comparison = pd.DataFrame({
    'Model'           : ['ANN', 'CNN', 'CNN + Augmentation'],
    'Test Accuracy'   : [f'{ann_test_acc*100:.2f}%', f'{cnn_test_acc*100:.2f}%', f'{aug_test_acc*100:.2f}%'],
    'Test Loss'       : [f'{ann_test_loss:.4f}', f'{cnn_test_loss:.4f}', f'{aug_test_loss:.4f}'],
    'Parameters'      : [f'{ann_params:,}', f'{cnn_params:,}', f'{aug_params:,}'],
    'Input Type'      : ['Flat vector', '2D spatial', '2D spatial + augmented'],
    'Spatial Aware'   : ['No', 'Yes', 'Yes'],
    'Augmentation'    : ['No', 'No', 'Yes'],
})

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print(comparison.to_string(index=False))
comparison

In [ ]:
# Bar chart comparison
models_list = ['ANN', 'CNN', 'CNN + Aug']
accuracies  = [ann_test_acc * 100, cnn_test_acc * 100, aug_test_acc * 100]
colors      = ['#4C72B0', '#55A868', '#C44E52']

plt.figure(figsize=(8, 5))
bars = plt.bar(models_list, accuracies, color=colors, edgecolor='black', width=0.5)

for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f'{acc:.2f}%',
        ha='center', va='bottom', fontweight='bold'
    )

plt.title('Model Accuracy Comparison on CIFAR-10', fontsize=13)
plt.ylabel('Test Accuracy (%)')
plt.ylim(0, 100)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 💾 Step 8: Save the Best Model

In [ ]:
# Save best model
aug_cnn_model.save('cifar10_best_model.keras')
print('Model saved as cifar10_best_model.keras')

# To reload:
# loaded_model = tf.keras.models.load_model('cifar10_best_model.keras')
# loaded_model.evaluate(x_test_norm, y_test)

## ✅ Conclusion

| What we learned | Key takeaway |
|---|---|
| ANN flattens images | Loses all spatial structure → lower accuracy |
| CNN uses Conv2D | Detects edges, textures, shapes → much better |
| BatchNormalization | Stabilizes & speeds up training |
| Dropout | Prevents overfitting |
| Data Augmentation | Improves generalization on unseen images |
| EarlyStopping | Stops training at best epoch automatically |
| ReduceLROnPlateau | Helps converge when loss stalls |

### 🎯 Next Steps (extend this project):
1. Try **transfer learning** with `MobileNetV2` or `ResNet50` — can reach 90%+
2. Use **learning rate warmup** scheduler
3. Add **label smoothing** to the loss function
4. Try **Mixup** or **CutMix** augmentation
5. Export model to **TFLite** for mobile deployment